In [1]:
import numpy as np 
inv = np.linalg.inv
tr = np.trace

In [2]:
def get_projector(ind, size):
    P = np.zeros((size,size))
    Q = np.eye(size)
    for i in ind:
        P[i,i] = 1
        Q[i,i] = 0
    return P, Q


def get_excited_matrix(A0, ind):
    N = A0.shape[0]
    AX = np.copy(A0)
    for i in ind:
        AX[:,i] = np.random.rand(N)
    return AX

In [3]:
size = 3
index = [0,2]
A0 = np.random.rand(size,size)
AX = get_excited_matrix(A0, index)

B0 = np.random.rand(size,size)
BX = get_excited_matrix(B0, index)

P, Q = get_projector(index, size)


##  Check for the corretness of the single-electron expression

$$\textnormal{Tr}\left( A_X^{-1}B_X\right) = \textnormal{Tr}\left( A_0^{-1}B_0\right) + \textnormal{Tr}\left([A_0^{-1}A_v]_X^{-1}  [M]_X  \right)$$

$$
M = A_0^{-1}B_v-A_0^{-1}B_0A_0^{-1}A_v     
$$


In [4]:
T = P @ inv ( inv(A0)@AX) @ P
M = inv(A0) @ BX - inv(A0) @ B0 @ inv(A0) @ AX

ref = tr(inv(AX) @BX) 
out = tr(inv(A0) @ B0) + tr (T @ P@M@P )

print(ref, out)

-0.8668942055345381 -0.8668942055345363



## Check for the correctess of he following expression 

$$ \textnormal{Tr} \left(A_X^{-1}B_XA_X^{-1}B_X\right) = \textnormal{Tr}\left( A_0^{-1}B_0A_0^{-1}B_0\right) + \textnormal{Tr}\left( [M]_X^2 \right) + 2\textnormal{Tr}\left([Y]_X)  \right) $$

$$
M = A_0^{-1}B_v-A_0^{-1}B_0A_0^{-1}A_v     
$$

$$
    Y = A_0^{-1} B_0 M
$$

In [5]:
ref = tr ( inv(AX)@BX@inv(AX)@BX )
Y = inv(A0) @B0 @ M
out = tr(inv(A0)@B0@inv(A0)@B0) + tr( T@P@M@M@P) + 2*tr(T@P@Y@P)
                                                      
print(ref, out)

-4.952077831765446 -21.232368825923587


### Derivation 
$$
 \textnormal{Tr}\left( (A_X^{-1} B_X)^2 \right) = \underbrace{\textnormal{Tr}\left( (A_0^{-1} B_X)^2 \right)}_\alpha + \underbrace{\textnormal{Tr}\left( (Z B_X)^2 \right)}_\gamma - 2 \underbrace{\textnormal{Tr}\left( A_0^{-1} B_X Z B_X \right)}_\beta
$$

with

$$
Z = \left(A_0^{-1} A_X - \mathbb{I}\right) T A_0^{-1}
$$

and 

$$
T =  \left(P (A_0^{-1}A_X)^{-1} P\right)
$$

In [6]:
ref = tr(inv(AX) @ BX @ inv(AX) @ BX) 
I = np.eye(size)

Z = (inv(A0) @ AX - I) @ T @ inv(A0)

alpha = tr(inv(A0) @ BX @ inv(A0) @BX)
gamma =  tr(Z @ BX @ Z @ BX)
beta = tr(inv(A0) @ BX @ Z @ BX)
out =  alpha + gamma - 2 * beta 
print(ref,out)

-4.952077831765446 -4.952077831765446


Unrolling the first term 
$$
\textnormal{Tr}\left( (A_0^{-1} B_X)^2 \right) =   \underbrace{\textnormal{Tr}\left( (A_0^{-1} B_0)^2 \right)}_{\alpha_1} +   \underbrace{\textnormal{Tr}\left( (A_0^{-1} B_\Delta)^2 \right)}_{\alpha_2} + 2 \underbrace{\textnormal{Tr}\left( A_0^{-1} B_\Delta A_0^{-1} B_0 \right)}_{\alpha_3}
$$
with
$$
B_\Delta = (B_X-B_0) P = B_X - B_0
$$

In [7]:
BD = (BX-B0)@P

alpha1 = tr(inv(A0) @ B0 @ inv(A0) @ B0) 
alpha2 = tr(inv(A0) @ BD @ inv(A0) @ BD)
alpha3 = tr(inv(A0) @ BD @ inv(A0) @ B0)

out = alpha1 + alpha2 +2 * alpha3

print(alpha, out)

-3.4416380865073988 -3.4416380865074103


The second term

$$
   \textnormal{Tr}\left( (Z B_X)^2 \right) = \underbrace{\textnormal{Tr}\left( (A_0^{-1} A_X T A_0^{-1} B_X)^2 \right)}_{\gamma_1} +  \underbrace{\textnormal{Tr}\left( (T A_0^{-1}  B_X)^2 \right)}_{\gamma_2} - 2 \underbrace{\textnormal{Tr}\left( (T A_0^{-1}  B_X)^2 A_0^{-1} A_X \right)}_{\gamma_3}
$$


In [8]:
ref = tr(Z@BX@Z@BX)

gamma1 = tr( inv(A0) @ AX @ T @ inv(A0) @ BX @ inv(A0) @ AX @ T @ inv(A0) @ BX)
gamma2 = tr ( T @ inv(A0) @ BX @ T @ inv(A0) @ BX)
gamma3 = tr ( ( T @ inv(A0) @ BX) @ ( T @ inv(A0) @ BX) @ inv(A0) @ AX)

print(ref, gamma1 + gamma2 - 2 * gamma3)

8.399085110668988 8.399085110668992


### Unrolling further

$$
   \gamma_1 = \underbrace{\textnormal{Tr}\left( (A_0^{-1} A_X T A_0^{-1} B_0)^2 \right)}_{\gamma_{11}} +  \underbrace{\textnormal{Tr}\left( (A_0^{-1} B_\Delta)^2 \right)}_{\gamma_{12}} + 2 \underbrace{\textnormal{Tr}\left( A_0^{-1} B_0 A_0^{-1} A_X T A_0^{-1} B_\Delta \right)}_{\gamma_{13}}
$$|

In [9]:
gamma11 = tr( (inv(A0)@AX@T@inv(A0)@B0) @ (inv(A0)@AX@T@inv(A0)@B0) )
gamma12 =  tr(inv(A0)@BD@inv(A0)@BD)
gamma13 = tr (  inv(A0) @ B0 @ inv(A0) @ AX @ T @ inv(A0) @ BD )

print(gamma1, gamma11+gamma12+2*gamma13)

37.67716507227982 37.67716507227986


$$
   \gamma_2 = \underbrace{\textnormal{Tr}\left( (T A_0^{-1}  B_0)^2 \right)}_{\gamma_{21}} + \underbrace{\textnormal{Tr}\left( (T A_0^{-1}  B_\Delta)^2 \right)}_{\gamma_{22}} + 2 \underbrace{\textnormal{Tr}\left( T A_0^{-1} B_0 T A_0^{-1} B_\Delta \right)}_{\gamma_{23}}
$$


In [10]:
gamma21 = tr ( (T@inv(A0)@B0) @ (T@inv(A0)@B0))
gamma22 = tr ( (T@inv(A0)@BD) @ (T@inv(A0)@BD))
gamma23 = tr(T @ inv(A0)@B0@T@inv(A0)@BD)

print(gamma2, gamma21 + gamma22 + 2*gamma23)

37.59723350286786 37.59723350286782


$$
   \gamma_3 =\overbrace{\textnormal{Tr}\left( (T A_0^{-1}  B_0)^2 A_0^{-1} A_X \right)}^{\gamma_{31}} + \overbrace{\textnormal{Tr}\left( (A_0^{-1}  B_\Delta)^2 T \right)}^{\gamma_{32}}
   + \underbrace{\textnormal{Tr}\left( A_0^{-1} B_0 T A_0^{-1} B_\Delta \right)}_{\gamma_{33}} + \underbrace{\textnormal{Tr}\left(T  A_0^{-1} B_\Delta T A_0^{-1} B_0 A_0^{-1} A_X \right)}_{\gamma_{34}}
$$

In [11]:
gamma31 = tr(  (T @ inv(A0)@B0) @ (T @ inv(A0)@B0) @inv(A0) @ AX )
gamma32 = tr( (inv(A0)@BD) @ (inv(A0)@BD) @ T )
gamma33 = tr( inv(A0) @ B0 @ T @ inv(A0) @ BD)
gamma34 = tr ( T @ inv(A0) @ BD @ T @ inv(A0) @ B0 @ inv(A0) @ AX)

print(gamma3, gamma31 + gamma32 + gamma33 + gamma34)

33.43765673223934 33.43765673223929


The third term

$$
     \textnormal{Tr}\left( A_0^{-1} B_X Z B_X \right) = \underbrace{\textnormal{Tr}\left( A_0^{-1} B_X A_0^{-1} A_X T A_0^{-1} B_X \right) }_{\beta_1} - \underbrace{\textnormal{Tr}\left( A_0^{-1} B_X T A_0^{-1} B_X \right) }_{\beta_2}
$$

In [12]:
ref = tr( inv(A0) @ BX @ Z @ BX)
beta1 = tr(inv(A0) @ BX @ inv(A0) @ AX @ T @ inv(A0) @ BX)
beta2 = tr(inv(A0) @ BX @ T @ inv(A0) @ BX)

print(ref, beta1 - beta2)

4.954762427963518 4.954762427963521


unrilling the tems

$$
    \beta_1 = \overbrace{\textnormal{Tr}\left( (A_0^{-1} B_0)^2  A_0^{-1} A_X T\right)}^{\beta_{11}} +  \overbrace{\textnormal{Tr}\left( (A_0^{-1} B_\Delta)^2 \right)}^{\beta_{12}}  
    +  \underbrace{\textnormal{Tr}\left( A_0^{-1} B_0 A_0^{-1} A_X T A_0^{-1} B_\Delta \right)}_{\beta_{13}} +  \underbrace{\textnormal{Tr}\left( A_0^{-1} B_\Delta A_0^{-1} B_0 \right)}_{\beta_{14}}
$$

In [13]:
beta11 = tr( inv(A0) @ B0 @ inv(A0) @ B0 @ inv(A0) @ AX @ T )
beta12 = tr (inv(A0) @ BD @ inv(A0) @ BD)
beta13 = tr ( inv(A0) @ B0 @ inv(A0) @ AX @ T @ inv(A0)  @ BD )
beta14 = tr ( inv(A0) @ BD @ inv(A0) @ B0 )

print(beta1, beta11 + beta12 + beta13 + beta14)

7.33699515626121 7.336995156261231


$$
    \beta_2 = \underbrace{\textnormal{Tr}\left( (A_0^{-1} B_0)^2 T \right)}_{\beta_{21}} + \underbrace{\textnormal{Tr}\left( (A_0^{-1} B_\Delta)^2 T \right)}_{\beta_{22}} + \underbrace{\textnormal{Tr}\left( A_0^{-1} B_\Delta A_0^{-1} B_0 T \right)}_{\beta_{23}} + \underbrace{\textnormal{Tr}\left( A_0^{-1} B_0 A_0^{-1} B_\Delta T \right)}_{\beta_{24}}
$$

In [14]:
beta21 = tr ( inv(A0)@B0@inv(A0)@B0@T )
beta22 = tr( inv(A0)@BD@inv(A0)@BD@T)
beta23 = tr( inv(A0)@BD@inv(A0)@B0@T)
beta24 = tr(inv(A0)@B0@inv(A0)@BD@T)
print(beta2, beta21+beta22+beta23+beta24)

2.382232728297689 2.382232728297673


### Facorization

See the equation below

$$
\alpha_2 + \gamma_{12} - 2 \beta_{12} = 0 \\
2 \alpha_3 - \beta_{13}  = 0 \\
\gamma_{32} - \beta_{22} = 0 \\
\gamma_{33}-\beta_{23} = 0
$$

Leads to 

$$ 
\textnormal{Tr} \left(A_X^{-1}B_XA_X^{-1}B_X\right) = \alpha_1  + \gamma_{11} + \gamma_2 - 2\gamma_{31} - 2\gamma_{34} - 2 \beta_{11} + 2 \beta_{21} + 2 \beta_{24}
$$

In [27]:
ref = tr(inv(AX) @ BX @ inv(AX) @ BX) 

print(alpha2 + gamma12 - 2 * beta12)
print(alpha3 - beta14)
print(gamma13 - beta13)
print(gamma32 - beta22)
print(gamma33 - beta23)

print(ref)
print(alpha1 + gamma11 + gamma2 -2*gamma31 -2*gamma34 - 2*beta11 + 2*beta21 + 2*beta24)

0.0
0.0
0.0
7.105427357601002e-15
-7.105427357601002e-15
-4.952077831765446
-4.9520778317653935


### further simpliciation 

by factorizing 

$$
B_X = B_0 + B_\Delta
$$

we can get

$$
\gamma_{31} + \gamma_{34} = \textnormal{Tr}\left( (T A_0^{-1}  B_0)^2 A_0^{-1} A_X \right) + \textnormal{Tr}\left(T  A_0^{-1} B_\Delta T A_0^{-1} B_0 A_0^{-1} A_X \right) = \textnormal{Tr}\left(T  A_0^{-1} B_X T A_0^{-1} B_0 A_0^{-1} A_X \right)
$$

In [25]:
print(gamma31 + gamma34 - tr((T@inv(A0)@BX)@(T@inv(A0)@B0)@inv(A0)@AX))

-4.263256414560601e-14


and also

$$
\beta_{21} + \beta_{24} = \textnormal{Tr}\left( (A_0^{-1} B_0)^2 T \right) + \textnormal{Tr}\left( A_0^{-1} B_0 A_0^{-1} B_\Delta T \right) = \textnormal{Tr}\left( A_0^{-1} B_0 A_0^{-1} B_X T \right) 
$$

In [26]:
print(beta21 + beta24 - tr(inv(A0)@B0@inv(A0)@BX@T))

-1.0658141036401503e-14


In [28]:
ref = tr(inv(AX) @ BX @ inv(AX) @ BX) 

gamma31_34 = tr((T@inv(A0)@BX)@(T@inv(A0)@B0)@inv(A0)@AX)
beta21_24 = tr(inv(A0)@B0@inv(A0)@BX@T)

print(ref)
print(alpha1 + gamma11 + gamma2 -2*gamma31_34 - 2*beta11 + 2*beta21_24)

-4.952077831765446
-4.95207783176545


$$ 
\textnormal{Tr} \left(A_X^{-1}B_XA_X^{-1}B_X\right) = \textnormal{Tr}\left( (A_0^{-1} B_0)^2 \right)  + \textnormal{Tr}\left( (A_0^{-1} A_X T A_0^{-1} B_0)^2 \right) + \textnormal{Tr}\left( (T A_0^{-1}  B_X)^2 \right) - 2 \textnormal{Tr}\left(T  A_0^{-1} B_X T A_0^{-1} B_0 A_0^{-1} A_X \right)
- 2 \textnormal{Tr}\left( (A_0^{-1} B_0)^2  A_0^{-1} A_X T\right) + 2 \textnormal{Tr}\left( A_0^{-1} B_0 A_0^{-1} B_X T \right)
$$

$$
\gamma_{11} + \gamma_{2} - 2 \gamma_{31|34} = \textnormal{Tr}\left( (T M)^2 \right)
$$

In [30]:
print(gamma11 + gamma2 -2*gamma31_34)
print(tr(T@M@T@M) )

0.6964740126644031
0.6964740126644173


$$
\beta_{11} - \beta_{21|24} = \textnormal{Tr}( A_0^{-1}B_0 M T)
$$

In [31]:
print(beta11 - beta21_24)
print(tr(inv(A0)@B0 @ M @ T))

13.948543403988698
-13.948543403988698


$$ 
\textnormal{Tr} \left(A_X^{-1}B_XA_X^{-1}B_X\right) = \textnormal{Tr}\left( (A_0^{-1} B_0)^2 \right)  + \textnormal{Tr}\left( (MT)^2 \right) + 2 \textnormal{Tr}\left(  A_0^{-1}B_0 M T\right)
$$

In [33]:
ref = tr(inv(AX) @ BX @ inv(AX) @ BX) 

x = tr(M@T@M@T)
y = tr(inv(A0)@B0 @ M @ T)
print(ref)
print(alpha1 + x + 2*y)

-4.952077831765446
-4.95207783176545
